In [ ]:
pip install transformers

In [ ]:
pip install datasets

In [ ]:
# TriviaQA

from datasets import load_dataset
import random
random.seed(10)

class QABenchmark:
    def __init__(self):
        self.dataset = []

    def sample(self, k: int):
        return random.sample(self.dataset, min(k, len(self.dataset)))

    def first_k(self, k: int):
        return self.dataset[:k]


class TriviaQA(QABenchmark):
    def __init__(self, split='validation'):
        super().__init__()
        loaded_dataset = load_dataset('trivia_qa', 'rc', split=split)
        self.dataset = [(example['question'], list(set([example['answer']['value']] + example['answer']['aliases'])))
                        for example in loaded_dataset]


class Lama(QABenchmark):
    def __init__(self, split: str = 'validation'):
        super().__init__()
        loaded_dataset = load_dataset('kilt_tasks', 'trex', split=f'{split}[:1000]')

        self.dataset = [
            (example['input'], example['output'][0]['answer'])
            for example in loaded_dataset
        ]


def get_optional_in_context_demonstrations_for_triviaqa(size: int = 200):
  trivia_qa_train_set = TriviaQA(split='train')
  return trivia_qa_train_set.first_k(k=size)


def get_triviaqa_validation_set(size: int = 100):
  trivia_qa_train_set = TriviaQA(split='validation')
  return trivia_qa_train_set.sample(k=size)


In [ ]:
# GPT2

import torch
from transformers import GPT2Tokenizer, GPT2LMHeadModel

def print_output(output: str):
    print("Output:\n" + 100 * '-')
    print(output)


def process_generation(text: str):
    if not text:
        return text
    while text and text[0] in ['\n', ':', ' ', ',', ';']:
        text = text[1:]
    return text


def load_gpt2(model_name: str = 'gpt2-medium'):
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name, pad_token_id=tokenizer.eos_token_id)
    return model, tokenizer


model, tokenizer = load_gpt2()

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model.to(device)


def sampling(input_text: str, max_length=50, temperature=0.7):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)
    input_ids_len = input_ids.shape[1]
    sample_output = model.generate(
        input_ids,
        do_sample=True,
        max_length=input_ids_len + max_length,
        top_k=0,
        temperature=temperature,
    )
    return process_generation(tokenizer.decode(sample_output[0][input_ids_len:], skip_special_tokens=True))


def beam_search(input_text: str, max_length=20):
    input_ids = tokenizer.encode(input_text, return_tensors='pt').to(device)
    input_ids_len = input_ids.shape[1]
    beam_output = model.generate(
        input_ids,
        max_length=input_ids_len + max_length,
        num_beams=5,
        no_repeat_ngram_size=2,
        early_stopping=True,
        # output_scores=True,
    )
    return process_generation(tokenizer.decode(beam_output[0][input_ids_len:], skip_special_tokens=True))

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/718 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.52G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

In [ ]:
# Evaluation

import pandas as pd

def normalize_text(s):
    """Removing articles and punctuation, and standardizing whitespace are all typical text processing steps."""
    import string, re

    def remove_articles(text):
        regex = re.compile(r"\b(a|an|the)\b", re.UNICODE)
        return re.sub(regex, " ", text)

    def white_space_fix(text):
        return " ".join(text.split())

    def remove_punc(text):
        exclude = set(string.punctuation)
        return "".join(ch for ch in text if ch not in exclude)

    def lower(text):
        return text.lower()

    return white_space_fix(remove_articles(remove_punc(lower(s))))


def compute_exact_match(prediction, truth):
    return int(normalize_text(prediction) == normalize_text(truth))


def check_answer_truthfulness(generated_answer, gold_answers):
    if isinstance(gold_answers, str):
        gold_answers = [gold_answers]
    normalized_generation = normalize_text(generated_answer)
    return any([normalize_text(answer) in normalized_generation for answer in gold_answers])

In [ ]:
optional_in_context_demonstrations = get_optional_in_context_demonstrations_for_triviaqa(size=500)
validation_set = get_triviaqa_validation_set(size=200)

README.md: 0.00B [00:00, ?B/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

rc/train-00000-of-00026.parquet:   0%|          | 0.00/308M [00:00<?, ?B/s]

rc/train-00001-of-00026.parquet:   0%|          | 0.00/298M [00:00<?, ?B/s]

...

Generating validation split:   0%|          | 0/17944 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/17210 [00:00<?, ? examples/s]

Loading dataset shards:   0%|          | 0/24 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/26 [00:00<?, ?it/s]

In [ ]:
def build_demo_block(demos):
  """
  input: demos - list of tuples (question:str, answers:list[str])
  output: formatted string with repeated: Question: ... Answer: ...
  """
  parts = []
  for q, answers in demos:
    ## choose one answer (the prompt must include only one answer)
    if isinstance(answers, list) and len(answers) > 0: ## safety only
      ans = answers[0]
    else:
      ans = str(answers)
    parts.append(f"Question: {q}\nAnswer: {ans}\n")

  return "".join(parts)

def build_full_prompt(demos, question):
  """
  input: demos (like before) and the new question we want the model to answer
  output: final prompt string for the GPT-2 to answer
  """
  demo_block = build_demo_block(demos)
  return demo_block + f"Question: {question}\nAnswer:" ##the last Answer: is empty so GPT-2 completes it

def evaluate_random_demos(k, decode_func):
  """
  input: k - num of demostrations, decode_func - beam_search or sampling(prompt, temperature=0.7)
  output: accuracy (float) - rate of success
  """
  demos = random.sample(optional_in_context_demonstrations, k)

  correct = 0
  total = 0

  for q, corr_answers in validation_set:
    prompt = build_full_prompt(demos, q)
    pred = decode_func(prompt)
    bool_correct = check_answer_truthfulness(pred, corr_answers)
    correct += int(bool_correct)
    total += 1

  return correct / total

## now we actually run the k sweep for beam search and store the results
ks = [3, 4, 5, 6, 7, 8]

beam_results = []

for k in ks:
  acc_rate = evaluate_random_demos(k, beam_search)
  beam_results.append({"k": k, "accuracy:": acc_rate})

beam_df = pd.DataFrame(beam_results)


##now we do the same for sampling with temp of 0.7

sample_results = []

for k in ks:
  acc_rate = evaluate_random_demos(k, lambda p: sampling(p, temperature=0.7))
  sample_results.append({"k":k, "accuracy:": acc_rate})

sample_df = pd.DataFrame(sample_results)

print("BEAM SEARCH RESULTS")
display(beam_df)

print("\nSAMPLING RESULTS (temp=0.7)")
display(sample_df)


The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


BEAM SEARCH RESULTS


,k,accuracy:
0,3,0.130
1,4,0.130
2,5,0.105
3,6,0.100
4,7,0.120
5,8,0.140



SAMPLING RESULTS (temp=0.7)


,k,accuracy:
0,3,0.080
1,4,0.050
2,5,0.045
3,6,0.055
4,7,0.070
5,8,0.075


In [ ]:
from transformers import AutoTokenizer, AutoModel

def cls_pooling(model_output, attention_mask):
    return model_output[0][:,0]

bert_tokenizer = AutoTokenizer.from_pretrained('sentence-transformers/bert-base-nli-cls-token')
bert_model = AutoModel.from_pretrained('sentence-transformers/bert-base-nli-cls-token')

def encode_question(question: str):
  encoded_input = bert_tokenizer([question], padding=True, truncation=True, return_tensors='pt')

  with torch.no_grad():
    model_output = bert_model(**encoded_input)

  # Perform pooling. In this case, max pooling.
  sentence_embeddings = cls_pooling(model_output, encoded_input['attention_mask'])

  return sentence_embeddings

tokenizer_config.json:   0%|          | 0.00/395 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/623 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

In [ ]:
## we go through each question, embedd it and store it to save computation time (200*500)
demo_embeddings = []
for demo_question, _ in optional_in_context_demonstrations:
  emb = encode_question(demo_question)
  demo_embeddings.append(emb)

def evaluate_retrieved_demos(decode_func):
    correct = 0
    total = 0

    for q, correct_answers in validation_set:
      ## encode validation question
      query_emb = encode_question(q)

      ## compute the similarity scores
      scores = []
      for demo_emb in demo_embeddings:
        score = torch.dot(query_emb.squeeze(), demo_emb.squeeze()) ## dot product - as required
        scores.append(score)

      ## select top 8 demos (based on scores)
      top_8_indices = torch.topk(torch.tensor(scores), k=8).indices.tolist()
      retrieved_demos = [optional_in_context_demonstrations[i] for i in top_8_indices]

      ##build the prompt
      prompt = build_full_prompt(retrieved_demos, q)

      ##generate answer
      pred = decode_func(prompt)
      is_correct = check_answer_truthfulness(pred, correct_answers)
      correct += int(is_correct)
      total += 1

    return correct / total
retrieved_beam_acc = evaluate_retrieved_demos(beam_search)
retrieved_sample_acc = evaluate_retrieved_demos(lambda p: sampling(p, temperature=0.7))
print(f"retrieved beam acc: {retrieved_beam_acc}")
print(f"retrieved sample acc: {retrieved_sample_acc}")


Asking to truncate to max_length but no maximum length is provided and the model has no predefined maximum length. Default to no truncation.


retrieved beam acc: 0.105
retrieved sample acc: 0.075


In [ ]:
lama_validation_set = Lama().sample(200)

README.md: 0.00B [00:00, ?B/s]

trex/train-00000-of-00003.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

trex/train-00001-of-00003.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

trex/train-00002-of-00003.parquet:   0%|          | 0.00/182M [00:00<?, ?B/s]

trex/validation-00000-of-00001.parquet:   0%|          | 0.00/1.19M [00:00<?, ?B/s]

trex/test-00000-of-00001.parquet:   0%|          | 0.00/490k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2284168 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/5000 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/5000 [00:00<?, ? examples/s]

In [ ]:
## we define a func to evaluate zero shot preformence of the GPT-2 with the triviaQA dataset
def eval_zero_shot_triviaqa(decode_func):
  correct = 0
  total = 0

  for q, correct_answer in validation_set:
    prompt = f"Question: {q}\nAnswer:"
    pred = decode_func(prompt)

    if check_answer_truthfulness(pred, correct_answer):
      correct +=1
    total +=1

  return correct / total


## we define a func to evaluate zero shot preformance of the GPT-2 with the lama dataset
def eval_zero_shot_lama(decode_func):
  correct = 0
  total = 0

  ## according to the lama format (no questions, only complete the sentences)
  for prefix, correct_label in lama_validation_set:
    prompt = prefix
    pred = decode_func(prompt)

    #check if the pred is right
    if normalize_text(correct_label) in normalize_text(pred):
      correct += 1
    total += 1

  return correct / total

## preform the actual evaluation on TriviaQA with beam search and sampling
zero_shot_triviaqa_beam = eval_zero_shot_triviaqa(beam_search)
zero_shot_triviaqa_sample = eval_zero_shot_triviaqa(lambda p: sampling(p, temperature=0.7))

print("Zero-shot TriviaQA (beam):", zero_shot_triviaqa_beam)
print("Zero-shot TriviaQA (sampling):", zero_shot_triviaqa_sample)

## preform the actual evaluation on LAMA with beam search and sampling
zero_shot_lama_beam = eval_zero_shot_lama(beam_search)
zero_shot_lama_sample = eval_zero_shot_lama(lambda p: sampling(p, temperature=0.7))

print("Zero-shot LAMA (beam):", zero_shot_lama_beam)
print("Zero-shot LAMA (sampling):", zero_shot_lama_sample)


Zero-shot TriviaQA (beam): 0.085
Zero-shot TriviaQA (sampling): 0.08
Zero-shot LAMA (beam): 0.09
Zero-shot LAMA (sampling): 0.05
